# Import Packages

I'll make use of the following packages:
- `numpy` is a package for scientific computing in python.
- `pandas` A powerful Python library for data manipulation and analysis.
- `seaborn` A data visualization library based on matplotlib.
- `scikit-learn` A comprehensive library for machine learning in Python.
- `kaggle` Using Kaggle API to download data.

In [67]:
import numpy as np
import pandas as pd
import kagglehub
import os
import seaborn as sns
import matplotlib.pyplot as plt
from collections import defaultdict


# Download Data

In [2]:
# Download latest version
path = kagglehub.dataset_download("taeefnajib/used-car-price-prediction-dataset")

print(os.listdir(path))

['used_cars.csv']


In [3]:
# Load the dataset
df = pd.read_csv(os.path.join(path, "used_cars.csv"))

# Data preprocessing

## Check data

In [4]:
df.head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,Ford,Utility Police Interceptor Base,2013,"51,000 mi.",E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$10,300"
1,Hyundai,Palisade SEL,2021,"34,742 mi.",Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,"$38,005"
2,Lexus,RX 350 RX 350,2022,"22,372 mi.",Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,"$54,598"
3,INFINITI,Q50 Hybrid Sport,2015,"88,900 mi.",Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,Yes,"$15,500"
4,Audi,Q3 45 S line Premium Plus,2021,"9,835 mi.",Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,"$34,999"


In [109]:
## check for missing values
df.isna().sum()

brand             0
model             0
model_year        0
milage            0
fuel_type       170
engine            0
transmission      0
ext_col           0
int_col           0
accident        113
clean_title     596
price             0
dtype: int64

## 📝 Initial Data Glance – Observations & Next Steps

### Overview

- **Columns:**  
  The dataset contains **12 columns**: `brand`, `model`, `model_year`, `milage`, `fuel_type`, `engine`, `transmission`, `ext_col`, `int_col`, `accident`, `clean_title`, `price`.

- **Data Types:**  
  Most columns are **object type** (categorical or text). Data transformation will be required (label encoding, one-hot encoding, parsing text fields).

- **Brand & Model:**  
  - Some `model` values are duplicated or concatenated (e.g., `RX 350 RX 350`).
  - Will inspect and **clean/split/merge brand and model** to ensure unique, consistent values.

- **Feature Extraction:**  
  - Columns such as `engine` and `transmission` contain multiple details (e.g., horsepower, engine size, cylinder count, speed type) that can be **parsed into new features**.

- **Missing Values:**  
  - Nulls detected in several columns (e.g., `clean_title`).  
  - Will analyze missingness and apply appropriate imputation (mode, new category, or predictive imputation).

- **Target Variable:**  
  - The `price` column is a string with currency symbol and commas—needs to be cleaned and converted to numeric.

- **Formatting Issues:**  
  - Fields like `milage` and `price` contain units/symbols (e.g., "mi.", "$", ",")—will remove for numeric conversion.
  - `accident` and `clean_title` are categorical but may need binarization or mapping.

---

### Next Steps

- Clean and standardize all categorical and text fields.
- Parse and extract features from `engine` and `transmission` columns.
- Handle missing values with suitable imputation strategies.
- Convert `price` and `milage` to numeric types.
- Ensure brand/model consistency for analysis and modeling.

## Column Transformation

### Brand and Model

In [68]:
brand_list = sorted(df['brand'].unique())

by_letter = defaultdict(list)
for brand in brand_list:
    by_letter[brand[0].upper()].append(brand)
    

for letter in by_letter.keys():
    print(f"{letter}:{', '.join(by_letter[letter])}")

A:Acura, Alfa, Aston, Audi
B:BMW, Bentley, Bugatti, Buick
C:Cadillac, Chevrolet, Chrysler
D:Dodge
F:FIAT, Ferrari, Ford
G:GMC, Genesis
H:Honda, Hummer, Hyundai
I:INFINITI
J:Jaguar, Jeep
K:Karma, Kia
L:Lamborghini, Land, Lexus, Lincoln, Lotus, Lucid
M:MINI, Maserati, Maybach, Mazda, McLaren, Mercedes-Benz, Mercury, Mitsubishi
N:Nissan
P:Plymouth, Polestar, Pontiac, Porsche
R:RAM, Rivian, Rolls-Royce
S:Saab, Saturn, Scion, Subaru, Suzuki, smart
T:Tesla, Toyota
V:Volkswagen, Volvo


---
🏷️ Brand Name Inconsistencies

While reviewing the car brands, I noticed that several are missing their full names:

- **Alfa** → _Alfa Romeo_
- **Aston** → _Aston Martin_
- **Land** → _Land Rover_

Let’s take a closer look at these brands to ensure correct and consistent naming throughout the dataset.

In [21]:
df[df['brand'] == 'Alfa'].head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
151,Alfa,Romeo Stelvio Ti Sport,2020,"18,665 mi.",Gasoline,2.0L I4 16V GDI SOHC Turbo,8-Speed Automatic,Lunare White Metallic,Ice,None reported,Yes,"$35,645"
255,Alfa,Romeo Giulia Quadrifoglio,2022,"1,966 mi.",Gasoline,2.9L V6 24V GDI DOHC Twin Turbo,8-Speed Automatic,Verde,Black,None reported,NaN,"$75,900"
343,Alfa,Romeo Stelvio Ti,2020,"41,000 mi.",Gasoline,280.0HP 2.0L 4 Cylinder Engine Gasoline Fuel,8-Speed A/T,White,Black,None reported,Yes,"$32,400"
412,Alfa,Romeo Stelvio Quadrifoglio,2019,"26,500 mi.",Gasoline,505.0HP 2.9L V6 Cylinder Engine Gasoline Fuel,8-Speed A/T,Gray,Black,None reported,Yes,"$53,900"
414,Alfa,Romeo Stelvio Ti Sport,2020,"21,487 mi.",Gasoline,2.0L I4 16V GDI SOHC Turbo,8-Speed Automatic,Anodized Blue Metallic,Ice,None reported,Yes,"$35,345"


In [22]:
df[df['brand'] == 'Aston'].head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
11,Aston,Martin DBS Superleggera,2019,"22,770 mi.",Gasoline,715.0HP 5.2L 12 Cylinder Engine Gasoline Fuel,8-Speed A/T,Silver,Black,None reported,Yes,"$184,606"
93,Aston,Martin DBS Superleggera,2021,"2,165 mi.",Gasoline,5.2L V12 48V GDI DOHC Twin Turbo,8-Speed Automatic,Black,Black,None reported,Yes,"$279,950"
314,Aston,Martin DBX Base,2021,"2,353 mi.",Gasoline,4.0L V8 32V GDI DOHC Twin Turbo,9-Speed Automatic,Green,Sahara Tan,None reported,Yes,"$159,500"
535,Aston,Martin V8 Vantage Base,2008,"25,025 mi.",Gasoline,380.0HP 4.3L 8 Cylinder Engine Gasoline Fuel,6-Speed A/T,White,Black,At least 1 accident or damage reported,Yes,"$39,000"
610,Aston,Martin V8 Vantage Base,2008,"62,378 mi.",Gasoline,380.0HP 4.3L 8 Cylinder Engine Gasoline Fuel,M/T,Red,Beige,At least 1 accident or damage reported,Yes,"$33,995"


In [23]:
df[df['brand'] == 'Land'].head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
10,Land,Rover Range Rover Sport 3.0 Supercharged HST,2021,"27,608 mi.",Gasoline,V6,Automatic,Fuji White,Pimento / Ebony,None reported,NaN,"$73,897"
15,Land,Rover LR4 HSE,2013,"79,800 mi.",Gasoline,375.0HP 5.0L 8 Cylinder Engine Gasoline Fuel,A/T,White,Black,None reported,Yes,"$29,990"
80,Land,Rover Discovery Sport SE R-Dynamic,2020,"21,240 mi.",Gasoline,2.0 Liter,Automatic,White,Black,None reported,NaN,"$37,998"
110,Land,Rover LR4 HSE LUX Landmark Edition,2016,"144,000 mi.",Gasoline,340.0HP 3.0L V6 Cylinder Engine Gasoline Fuel,8-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$18,000"
120,Land,Rover Range Rover Sport 3.0L Supercharged HSE,2018,"104,700 mi.",Gasoline,V6,Automatic,Fuji White,Ivory / Ebony,At least 1 accident or damage reported,NaN,"$30,775"


---
### 🔧 Brand Name Corrections Needed

As predicted, these three brands require their brand and model names to be fixed for consistency:

- **Alfa** → _Alfa Romeo_
- **Aston** → _Aston Martin_
- **Land** → _Land Rover_

In [47]:
### Fixing Brand Name Inconsistencies

df_update = df.copy()

## Replace inconsistent brand names with full names
df_update['brand'] = df_update['brand'].replace({
    'Alfa': 'Alfa Romeo',
    'Aston': 'Aston Martin',
    'Land': 'Land Rover'
})

## Remove first word from model names for these brands
brand_name = ['Alfa Romeo', 'Aston Martin', 'Land Rover']

for brand in brand_name:

    df_update.loc[df_update['brand'] == brand, 'model'] = df_update.loc[df_update['brand'] == brand, 'model'].str.split().apply(lambda x: ' '.join(x[1:]) if isinstance(x, list) and len(x) > 1 else '')


In [48]:
df_update[df_update['brand'] == 'Alfa Romeo'].head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
151,Alfa Romeo,Stelvio Ti Sport,2020,"18,665 mi.",Gasoline,2.0L I4 16V GDI SOHC Turbo,8-Speed Automatic,Lunare White Metallic,Ice,None reported,Yes,"$35,645"
255,Alfa Romeo,Giulia Quadrifoglio,2022,"1,966 mi.",Gasoline,2.9L V6 24V GDI DOHC Twin Turbo,8-Speed Automatic,Verde,Black,None reported,NaN,"$75,900"
343,Alfa Romeo,Stelvio Ti,2020,"41,000 mi.",Gasoline,280.0HP 2.0L 4 Cylinder Engine Gasoline Fuel,8-Speed A/T,White,Black,None reported,Yes,"$32,400"
412,Alfa Romeo,Stelvio Quadrifoglio,2019,"26,500 mi.",Gasoline,505.0HP 2.9L V6 Cylinder Engine Gasoline Fuel,8-Speed A/T,Gray,Black,None reported,Yes,"$53,900"
414,Alfa Romeo,Stelvio Ti Sport,2020,"21,487 mi.",Gasoline,2.0L I4 16V GDI SOHC Turbo,8-Speed Automatic,Anodized Blue Metallic,Ice,None reported,Yes,"$35,345"


In [51]:
df_update[df_update['brand'] == 'Aston Martin'].head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
11,Aston Martin,DBS Superleggera,2019,"22,770 mi.",Gasoline,715.0HP 5.2L 12 Cylinder Engine Gasoline Fuel,8-Speed A/T,Silver,Black,None reported,Yes,"$184,606"
93,Aston Martin,DBS Superleggera,2021,"2,165 mi.",Gasoline,5.2L V12 48V GDI DOHC Twin Turbo,8-Speed Automatic,Black,Black,None reported,Yes,"$279,950"
314,Aston Martin,DBX Base,2021,"2,353 mi.",Gasoline,4.0L V8 32V GDI DOHC Twin Turbo,9-Speed Automatic,Green,Sahara Tan,None reported,Yes,"$159,500"
535,Aston Martin,V8 Vantage Base,2008,"25,025 mi.",Gasoline,380.0HP 4.3L 8 Cylinder Engine Gasoline Fuel,6-Speed A/T,White,Black,At least 1 accident or damage reported,Yes,"$39,000"
610,Aston Martin,V8 Vantage Base,2008,"62,378 mi.",Gasoline,380.0HP 4.3L 8 Cylinder Engine Gasoline Fuel,M/T,Red,Beige,At least 1 accident or damage reported,Yes,"$33,995"


In [53]:
df_update[df_update['brand'] == 'Land Rover'].head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
10,Land Rover,Range Rover Sport 3.0 Supercharged HST,2021,"27,608 mi.",Gasoline,V6,Automatic,Fuji White,Pimento / Ebony,None reported,NaN,"$73,897"
15,Land Rover,LR4 HSE,2013,"79,800 mi.",Gasoline,375.0HP 5.0L 8 Cylinder Engine Gasoline Fuel,A/T,White,Black,None reported,Yes,"$29,990"
80,Land Rover,Discovery Sport SE R-Dynamic,2020,"21,240 mi.",Gasoline,2.0 Liter,Automatic,White,Black,None reported,NaN,"$37,998"
110,Land Rover,LR4 HSE LUX Landmark Edition,2016,"144,000 mi.",Gasoline,340.0HP 3.0L V6 Cylinder Engine Gasoline Fuel,8-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$18,000"
120,Land Rover,Range Rover Sport 3.0L Supercharged HSE,2018,"104,700 mi.",Gasoline,V6,Automatic,Fuji White,Ivory / Ebony,At least 1 accident or damage reported,NaN,"$30,775"


---

Now that I’ve fixed the brand names, let’s address another issue: **duplicate model names**.  
For example, in row 2 I saw `Lexus RX 350 RX 350` as a model. These duplicates inflate the number of unique model groups.

**Next step:**  
Clean the `model` column to remove repeated names and reduce redundancy in our model grouping.

### 🔧Clean Model names

In [ ]:
# Remove duplicate model names
# For example, 'RX 350 RX 350' should be 'RX 350'
df_update['model'] = df_update['model'].apply(lambda x: ' '.join(dict.fromkeys(x.split())))

### Extract numbers columns

In [ ]:
col_names = ['milage', 'price']

def clean_numeric_col(series):
    """
    Clean numeric columns by removing non-numeric characters and converting to float.
    """
    return (
        series.astype(str)  # Ensure the series is of string type
        .str.replace(r'[^\d.]', '', regex=True)  # Remove non-numeric characters except digits and "."
        .astype(float)  # Convert to float
    )

# Apply the cleaning function to the specified columns
for col in col_names:
    df_update[col] = clean_numeric_col(df[col])

🔢 Now, both `milage` and `price` have been successfully converted to numerical columns.

---

### Individual columns
#### ⛽ Fuel Type: Data Cleaning Needed

In [81]:
df_update['fuel_type'].unique()

array(['E85 Flex Fuel', 'Gasoline', 'Hybrid', nan, 'Diesel',
       'Plug-In Hybrid', '–', 'not supported'], dtype=object)

The `fuel_type` column contains multiple categories, null values, and some unsupported or placeholder entries (e.g., `nan`, `'–'`, `'not supported'`).  
I’ll need to clean and unify these values for consistent analysis and modeling.

In [115]:
mask = df_update['fuel_type'].isin([np.nan,'–', 'not supported']) | df_update['brand'].isna()
df.loc[mask, ['brand', 'model', 'fuel_type']]['brand'].value_counts()

brand
Tesla            87
Rivian           17
Ford             17
Porsche          11
Chevrolet         8
Nissan            8
Dodge             8
Toyota            6
Audi              6
Mercedes-Benz     5
BMW               5
Mazda             4
Kia               4
Cadillac          4
Chrysler          3
Volkswagen        3
Hyundai           3
Lucid             3
Acura             2
Karma             2
Volvo             2
GMC               2
Rolls-Royce       1
FIAT              1
Honda             1
Mercury           1
Polestar          1
Jaguar            1
Jeep              1
Name: count, dtype: int64

Tesla, Lucid, and Rivian are pure electric brands—missing `fuel_type` values for these can be safely filled as `'Electric'`.  
For all other brands, I’ll need to inspect the model before imputing the correct fuel type.

In [116]:
mask = (df_update['brand'].isin(['Tesla', 'Lucid', 'Rivian']) & 
        (df_update['fuel_type'].isin([np.nan, '–', 'not supported']) |
         df_update['fuel_type'].isna()))

# Fill missing fuel_type for electric brands
df_update.loc[mask, 'fuel_type'] = 'Electric'

In [134]:
# update fuel_type for other brands

# Mapping dictionary: brand, model -> fuel type
fuel_type_map = {
    # --- Electric ---
    ("Porsche", "Taycan Base"): "Electric",
    ("Ford", "Mustang Mach-E GT"): "Electric",
    ("Audi", "e-tron Prestige"): "Electric",
    ("Ford", "Mustang Mach-E Premium"): "Electric",
    ("Porsche", "Taycan Turbo"): "Electric",
    ("Chevrolet", "Bolt EUV Premier"): "Electric",
    ("Chevrolet", "Bolt EV LT"): "Electric",
    ("Nissan", "Leaf SL"): "Electric",
    ("Nissan", "Leaf SV PLUS"): "Electric",
    ("Ford", "Mustang Mach-E Select"): "Electric",
    ("Mercedes-Benz", "EQS 450+ Base"): "Electric",
    ("Hyundai", "Kona EV SEL"): "Electric",
    ("Volkswagen", "ID.4 Pro S"): "Electric",
    ("Kia", "EV6 Wind"): "Electric",
    ("Kia", "EV6 GT-Line"): "Electric",
    ("Mercedes-Benz", "EQS 450 4MATIC"): "Electric",
    ("Polestar", "2 Launch Edition"): "Electric",
    ("Porsche", "Taycan"): "Electric",
    ("Porsche", "Taycan 4S"): "Electric",
    ("Toyota", "bZ4X Limited"): "Electric",
    ("Volkswagen", "e-Golf SE"): "Electric",
    ("Hyundai", "IONIQ 5 SE"): "Electric",
    ("Kia", "Niro EV EX"): "Electric",
    ("Audi", "Q4 e-tron 50 Premium Plus"): "Electric",
    ("Audi", "Q4 e-tron Sportback Premium"): "Electric",
    ("Audi", "e-tron Premium"): "Electric",
    ("BMW", "i3 94 Ah"): "Electric",
    ("BMW", "i3 Base"): "Electric",
    ("Cadillac", "LYRIQ Luxury"): "Electric",
    ("FIAT", "500e Battery Electric"): "Electric",
    ("Ford", "F-150 Lightning LARIAT"): "Electric",
    ("Ford", "F-150 Lightning XLT"): "Electric",
    ("Ford", "Mustang Mach-E California Route 1"): "Electric",
    ("GMC", "HUMMER EV Edition 1"): "Electric",
    ("Volvo", "C40 Recharge Pure Electric Twin Ultimate"): "Electric",
    ("Nissan", "Leaf S"): "Electric",

    # --- Hydrogen ---
    ("Toyota", "Mirai Base"): "Hydrogen",
    ("Toyota", "Mirai Limited"): "Hydrogen",

    # --- Plug-in Hybrid/Range Extender ---
    ("BMW", "i3 Base w/Range Extender"): "Plug-In Hybrid",
    ("BMW", "i3 120Ah w/Range Extender"): "Plug-In Hybrid",
    ("Karma", "Revero Base"): "Plug-In Hybrid",

    # --- Diesel ---
    ("Mercedes-Benz", "E-Class D 2.5 Turbo"): "Diesel",

    # --- Gasoline ---
    ("Cadillac", "DeVille Base"): "Gasoline",
    ("Toyota", "Land Cruiser Base"): "Gasoline",
    ("Dodge", "Challenger R/T Scat Pack"): "Gasoline",
    ("Dodge", "Challenger R/T"): "Gasoline",
    ("Chrysler", "Pacifica Touring"): "Gasoline",
    ("Ford", "Mustang EcoBoost Premium"): "Gasoline",
    ("Dodge", "Challenger SRT8 392"): "Gasoline",
    ("Dodge", "Challenger SRT 392"): "Gasoline",
    ("Dodge", "Challenger SRT8"): "Gasoline",
    ("Dodge", "Ram 3500 Quad Cab DRW"): "Gasoline",
    ("Chevrolet", "1500 Cheyenne"): "Gasoline",
    ("Chevrolet", "1500 Cheyenne Extended Cab"): "Gasoline",
    ("Chevrolet", "Sonic LT"): "Gasoline",
    ("Chrysler", "200 Limited"): "Gasoline",
    ("Ford", "Bronco"): "Gasoline",
    ("Ford", "Bronco XLT"): "Gasoline",
    ("Ford", "F-250 XL SuperCab H/D"): "Gasoline",
    ("GMC", "Sierra 1500 SLE1 Extended Cab"): "Gasoline",
    ("Honda", "Civic EX"): "Gasoline",
    ("Acura", "NSX Base"): "Gasoline",
    ("Jaguar", "XJ6 Vanden Plas"): "Gasoline",
    ("Jeep", "Wrangler S"): "Gasoline",
    ("Mazda", "Protege DX"): "Gasoline",
    ("Mazda", "Mazda6 i Grand Touring"): "Gasoline",
    ("Mercury", "Capri XR2"): "Gasoline",
    ("Acura", "Integra GS-R"): "Gasoline",
    ("Porsche", "911 Carrera"): "Gasoline",
    ("Nissan", "300ZX Base"): "Gasoline",
    ("Nissan", "Pickup Truck XE"): "Gasoline",
    ("Mazda", "MX-5 Miata Base"): "Gasoline",
    ("Porsche", "911 Carrera Cabriolet"): "Gasoline",
    ("Rolls-Royce", "Phantom"): "Gasoline",
    ("Volvo", "850 Turbo"): "Gasoline",
    ("Mazda", "Mazda3 s Grand Touring"): "Gasoline",
    ("Mercedes-Benz", "E-Class 400E"): "Gasoline",
    ("Nissan", "240SX Base"): "Gasoline",
}

def assign_fuel_type(row):
    if pd.isna(row['fuel_type']) or row['fuel_type'] in [np.nan, '–', 'not supported']:
        key = (row['brand'], row['model'])

        return fuel_type_map.get(key, row['fuel_type'])
    else:
        return row['fuel_type']
    
# Apply the mapping to the DataFrame
df_update['fuel_type'] = df_update.apply(assign_fuel_type, axis=1)


In [136]:
df_update['fuel_type'].unique()

array(['E85 Flex Fuel', 'Gasoline', 'Hybrid', 'Electric', 'Diesel',
       'Plug-In Hybrid', 'Hydrogen'], dtype=object)

✅ Fuel Type Cleanup Complete

All missing and inconsistent `fuel_type` values have been handled.  
The column is now clean and ready for analysis and modeling.

---